# “Does it run?” — from raw CAO data to a portfolio answer

This notebook answers one operational question for **every catalog campaign**:

> **Does the campaign currently run?**

It is a worked example of [`specs/computations.md`](../specs/computations.md), using the latest Activity JSONL snapshot.

The notebook does four things:

1. establishes how complete the raw evidence is;
2. resolves campaign, workflow, role, and target identity without confusing review-output routing with target selection;
3. prepares newest-first evaluation partitions; and
4. calls the same versioned `does-it-run` runtime intended for the dashboard data worker.

The notebook does **not** implement the health algorithm. That logic lives in `research/computations/does-it-run.js`, so notebook results and future dashboard results use the same semantics.

> Only bounded, non-sensitive Run fields are retained. Missing enrichment remains unknown. Raw prompts, transcripts, tool arguments, response bodies, and logs are never loaded into the analysis tables.


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import re
import subprocess
import time

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'activity/cao.mjs').exists()),
    None,
)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the gh-aw-cao repository')

SHARDS = ROOT / '.cao/gh-aw-logs-shards'
RUNTIME_BRIDGE = ROOT / 'research/computations/compute-does-it-run.mjs'
REFRESH_DATA = True
MAX_DISPLAY_ROWS = 100

pd.set_option('display.max_colwidth', 90)
print('Repository:', ROOT)


Repository: /Users/mnkiefer/gh-aw-cao


## 1. Establish the evidence boundary

The Activity snapshot contains both raw workflow Runs and Runs enriched by gh-aw. The `does-it-run` measure needs enriched `kind: run` records to associate workflow evidence with campaign semantics.

`audit-jsonl` reports coverage before any health conclusion is made. Partial evidence does not become a healthy or failed answer by default.


In [2]:
def cao(*arguments):
    completed = subprocess.run(
        ['node', 'activity/cao.mjs', *arguments],
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    return json.loads(completed.stdout)

if REFRESH_DATA:
    download = cao('download')

audit = cao('audit-jsonl')
source = audit['source']
canonical = audit['canonical']
evidence_boundary = pd.DataFrame([{
    'shards': audit['shards'],
    'unique raw Runs': source['uniqueRawRuns'],
    'unique enriched Runs': source['uniqueEnrichedRuns'],
    'enrichment coverage %': source['enrichmentCoveragePercent'],
    'canonical Runs': canonical['runs'],
    'canonical Audits': canonical['audits'],
}])

display(evidence_boundary)
print('Every result below is limited to enriched kind:run evidence.')


,shards,unique raw Runs,unique enriched Runs,enrichment coverage %,canonical Runs,canonical Audits
0,6,31951,2915,9.1,31951,85891


Every result below is limited to enriched kind:run evidence.


## 2. Resolve campaign identity and expected targets

The raw data tells us what was observed. Checked-in campaign records and control policy tell us what was expected.

Resolution rules:

1. workflow identity and role come from each campaign’s `cao.json`;
2. explicit policy `targets` take precedence;
3. a review-mode campaign without explicit targets expects the CAO control repository (`githubnext/gh-aw-cao`); and
4. another observed repository is retained as `observed-extra`, but it does not determine configured campaign health.

This prevents observed activity from silently widening rollout policy.


In [3]:
registry = []
for campaign_file in sorted(ROOT.glob('*/cao.json')):
    definition = json.loads(campaign_file.read_text())
    campaign = definition['campaign']
    orchestrator = definition['orchestrator']
    registry.append({
        'campaign': campaign,
        'role': 'orchestrator',
        'workflow': orchestrator,
        'workflow_path': f'.github/workflows/{orchestrator}.lock.yml',
    })
    for worker_role, workflow in definition.get('workers', {}).items():
        registry.append({
            'campaign': campaign,
            'role': 'worker',
            'worker_role': worker_role,
            'workflow': workflow,
            'workflow_path': f'.github/workflows/{workflow}.lock.yml',
        })

registry_df = pd.DataFrame(registry)
workflow_by_path = registry_df.set_index('workflow_path').to_dict('index')

policy = json.loads((ROOT / '.github/workflows/cao.json').read_text())
configurations = policy.get('control-plane', {}).get('campaigns', {})
expected_targets = {}
target_resolution = {}

for campaign in sorted(registry_df.campaign.unique()):
    raw_configuration = configurations.get(campaign, {})
    configuration = raw_configuration if isinstance(raw_configuration, dict) else {}
    explicit_targets = set((configuration.get('targets') or {}).keys())
    if explicit_targets:
        expected_targets[campaign] = explicit_targets
        target_resolution[campaign] = 'explicit-policy'
    else:
        expected_targets[campaign] = set()
        target_resolution[campaign] = 'dynamic-discovery'

inventory = registry_df.groupby(['campaign', 'role']).size().unstack(fill_value=0).reset_index()
inventory['expected targets'] = inventory.campaign.map(lambda value: len(expected_targets[value]))
inventory['target resolution'] = inventory.campaign.map(target_resolution)
display(inventory)


role,campaign,orchestrator,worker,expected targets,target resolution
0,cao-evolution,1,6,0,dynamic-discovery
1,dependabot,1,1,1,explicit-policy
2,eslint-rules,1,5,0,dynamic-discovery
3,eu-cra-compliance,1,6,0,dynamic-discovery
4,optimization,1,7,0,dynamic-discovery
5,repo-assist,1,4,0,dynamic-discovery
6,self-care,1,15,1,explicit-policy
7,software-development-practices,1,2,0,dynamic-discovery
8,uk-ai-advisory,1,1,0,dynamic-discovery


## 3. Read and deduplicate enriched Runs

Each JSONL shard is read once. The notebook keeps only fields needed to identify a Run, order it, evaluate its state, group failures, and link back to evidence.

Worker target names are extracted from producer titles shaped as `… · OWNER/REPOSITORY · MODE`. A missing target remains unknown rather than being assigned to an observed repository.


In [4]:
target_pattern = re.compile(r'^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$')


def target_from_name(name):
    parts = [part.strip() for part in (name or '').split('·')]
    target = next((part for part in parts[1:] if target_pattern.fullmatch(part)), None)
    mode = next(
        (part.lower() for part in reversed(parts[1:]) if part.lower() in {'live', 'review', 'preview'}),
        None,
    )
    return target, mode


def run_date(run):
    return run.get('started_at') or run.get('created_at') or run.get('updated_at')


started_at = time.perf_counter()
lines_read = 0
run_observations = 0
matched_observations = 0
deduplicated = {}

for shard in sorted(SHARDS.glob('*.jsonl')):
    with shard.open('rb') as handle:
        for line in handle:
            lines_read += 1
            if b'"kind":"run"' not in line[:80]:
                continue
            run_observations += 1
            run = json.loads(line).get('run') or {}
            workflow = workflow_by_path.get(run.get('workflow_path'))
            if workflow is None:
                continue
            matched_observations += 1
            target, mode = target_from_name(run.get('workflow_name')) if workflow['role'] == 'worker' else (None, None)
            row = {
                **workflow,
                'execution_repository': run.get('repository'),
                'target_repository': target,
                'rollout_mode': mode,
                'run_id': run.get('run_id'),
                'attempt': int(run.get('run_attempt') or 1),
                'status': (run.get('status') or 'unknown').lower(),
                'conclusion': (run.get('conclusion') or '').lower() or None,
                'classification': run.get('classification'),
                'failure_kind': run.get('failure_kind'),
                'run_date': run_date(run),
                'updated_at': run.get('updated_at'),
                'url': run.get('url'),
            }
            key = (row['execution_repository'], row['run_id'], row['attempt'])
            previous = deduplicated.get(key)
            if previous is None or (row['updated_at'] or '') >= (previous['updated_at'] or ''):
                deduplicated[key] = row

runs = pd.DataFrame(deduplicated.values())
for field in ('run_date', 'updated_at'):
    runs[field] = pd.to_datetime(runs[field], utc=True, errors='coerce')

load_metrics = pd.DataFrame([{
    'JSONL lines read': lines_read,
    'enriched Run observations': run_observations,
    'campaign Run observations': matched_observations,
    'distinct campaign Run attempts': len(runs),
    'declared workflows observed': runs.workflow.nunique(),
    'load seconds': round(time.perf_counter() - started_at, 2),
}])

display(load_metrics)
display(runs.groupby(['campaign', 'role']).size().rename('distinct Runs').reset_index())


,JSONL lines read,enriched Run observations,campaign Run observations,distinct campaign Run attempts,declared workflows observed,load seconds
0,146091,74024,57762,2213,45,5.3


,campaign,role,distinct Runs
0,cao-evolution,orchestrator,17
1,cao-evolution,worker,70
2,dependabot,orchestrator,84
3,dependabot,worker,64
4,eslint-rules,orchestrator,8
5,eslint-rules,worker,6
6,eu-cra-compliance,orchestrator,17
7,eu-cra-compliance,worker,369
8,optimization,orchestrator,97
9,optimization,worker,980


## 4. Compute `does-it-run` with the shared runtime

The computation receives **evaluation partitions**, not the whole raw dataset:

- one partition per campaign orchestrator workflow; and
- one partition per campaign worker workflow and target repository.

Each partition is ordered newest-first before it crosses the runtime boundary. The runtime then:

1. evaluates orchestrators first;
2. blocks worker evaluation if an orchestrator currently fails;
3. marks worker evaluation indeterminate if orchestrator evidence is missing or unknown;
4. otherwise walks each worker-target partition until its first successful Run;
5. groups only terminal failures newer than that success; and
6. excludes `observed-extra` targets from fixed-target campaign health.

Configured workers are not expanded across every target. A worker-target partition exists only when a Run is observed; an empty `not-observed` partition requires authoritative evidence that the exact invocation was required or dispatched. This snapshot does not yet expose that dispatch-intent contract, so intentionally skipped and conditional workers do not become synthetic failures. Review-mode output routing never supplies target identity.

Answer meanings:

| Answer | Meaning |
|---|---|
| `yes` | A success is the current boundary and no newer failure needs attention. |
| `no` | At least one terminal non-success is newer than the latest success. |
| `running` | Recovery activity is newer than the latest success and no newer terminal failure exists. |
| `not-observed` | No Run was retained for an expected partition. |
| `unknown` | Available evidence is insufficient for a reliable yes/no answer. |

The Python code below only adapts evidence into the input contract and presents the returned records. It contains no copy of the answer, gating, boundary, grouping, or rollup rules.


In [5]:
def json_scalar(value):
    if value is None or (not isinstance(value, (list, dict)) and pd.isna(value)):
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value.item() if hasattr(value, 'item') else value


def runtime_run(row):
    return {
        'id': f"github:run:{row['run_id']}:attempt:{row['attempt']}",
        'githubRunId': json_scalar(row['run_id']),
        'attempt': int(row['attempt']),
        'status': row['status'],
        'conclusion': json_scalar(row['conclusion']),
        'classification': json_scalar(row['classification']),
        'failureKind': json_scalar(row['failure_kind']),
        'startedAt': json_scalar(row['run_date']),
        'updatedAt': json_scalar(row['updated_at']),
        'runLink': json_scalar(row['url']),
    }


def make_partition(campaign, workflow, role, frame, target=None, membership=None):
    ordered = frame.sort_values(
        ['run_date', 'run_id', 'attempt'],
        ascending=[False, False, False],
        na_position='last',
    )
    return {
        'campaignId': campaign,
        'workflowId': workflow,
        'workflowRole': role,
        'workflowState': 'active',
        'targetRepositoryId': target,
        'targetScopeMembership': membership,
        'orderedNewestFirst': True,
        'runs': [runtime_run(row) for row in ordered.to_dict('records')],
    }


campaign_inputs = []
for campaign in sorted(registry_df.campaign.unique()):
    definitions = registry_df[registry_df.campaign == campaign]
    orchestrator_partitions = []
    worker_partitions = []

    for definition in definitions[definitions.role == 'orchestrator'].to_dict('records'):
        matching = runs[(runs.campaign == campaign) & (runs.workflow == definition['workflow'])]
        orchestrator_partitions.append(make_partition(
            campaign,
            definition['workflow'],
            'orchestrator',
            matching,
        ))

    expected = expected_targets[campaign]
    for definition in definitions[definitions.role == 'worker'].to_dict('records'):
        matching = runs[(runs.campaign == campaign) & (runs.workflow == definition['workflow'])]
        observed = set(matching.target_repository.dropna().unique())
        targets = sorted(observed)
        if matching.target_repository.isna().any():
            targets.append(None)
        for target in targets:
            target_runs = (
                matching[matching.target_repository == target]
                if target is not None
                else matching[matching.target_repository.isna()]
            )
            membership = 'expected' if target in expected else 'observed-extra' if expected else 'unknown'
            worker_partitions.append(make_partition(
                campaign,
                definition['workflow'],
                'worker',
                target_runs,
                target,
                membership,
            ))

    campaign_inputs.append({
        'campaignId': campaign,
        'orchestratorPartitions': orchestrator_partitions,
        'workerPartitions': worker_partitions,
    })

runtime_request = {'campaigns': campaign_inputs, 'benchmarkIterations': 30}
completed = subprocess.run(
    ['node', str(RUNTIME_BRIDGE)],
    cwd=ROOT,
    input=json.dumps(runtime_request),
    text=True,
    capture_output=True,
    check=True,
)
runtime_response = json.loads(completed.stdout)
runtime_result = runtime_response['result']
runtime_benchmark = runtime_response['benchmark']

partition_rows = []
for result in runtime_result['partitionResults']:
    latest_run = result.get('latestRun') or {}
    latest_success = result.get('latestSuccess') or {}
    partition_rows.append({
        'campaign': result['campaignId'],
        'workflow': result['workflowId'],
        'role': result['workflowRole'],
        'target_repository': result.get('targetRepositoryId'),
        'target_scope_membership': result.get('targetScopeMembership'),
        'answer': result['answer'],
        'needs_attention': result['needsAttention'],
        'latest_run_id': latest_run.get('githubRunId'),
        'latest_run_at': latest_run.get('observedAt'),
        'latest_run_url': latest_run.get('url'),
        'latest_success_id': latest_success.get('githubRunId'),
        'latest_success_at': latest_success.get('observedAt'),
        'runs_since_success': result['runsSinceSuccess'],
        'terminal_non_successes': result['terminalNonSuccessesSinceSuccess'],
        'active_runs': result['activeRunsSinceSuccess'],
        'unknown_date_runs': result['unknownDateRunCount'],
        'records_visited': result['recordsVisited'],
        'partition_runs_retained': result['partitionRunCount'],
    })
partitions_df = pd.DataFrame(partition_rows)

error_rows = []
for group in runtime_result['errorGroups']:
    latest_reference = group['runReferences'][0] if group['runReferences'] else {}
    error_rows.append({
        'campaign': group['campaignId'],
        'workflow': group['workflowId'],
        'role': group['workflowRole'],
        'target_repository': group.get('targetRepositoryId'),
        'target_scope_membership': group.get('targetScopeMembership'),
        'error_key': group['errorKey'],
        'count': group['count'],
        'latest_observed_at': group['latestObservedAt'],
        'latest_run_id': latest_reference.get('githubRunId'),
        'latest_run_url': latest_reference.get('url'),
        'current_run_ids': [reference.get('githubRunId') for reference in group['runReferences']],
        'current_run_urls': [reference.get('url') for reference in group['runReferences']],
    })
errors_df = pd.DataFrame(error_rows)

campaign_rows = []
for result in runtime_result['campaignResults']:
    campaign = result['campaignId']
    orchestrators = partitions_df[
        (partitions_df.campaign == campaign) & (partitions_df.role == 'orchestrator')
    ]
    campaign_rows.append({
        'campaign': campaign,
        'answer': result['answer'],
        'worker_evaluation_state': result['workerEvaluationState'],
        'orchestrator_answer': orchestrators.answer.iloc[0] if len(orchestrators) else 'unknown',
        'worker_partitions_evaluated': result['workerPartitionCount'],
        'attention_partitions': result['attentionPartitionCount'],
        'explicit_target_count': len(expected_targets[campaign]),
        'target_coverage': 'not-assessed-without-dispatch-intent',
        'records_visited': result['recordsVisited'],
        'partition_runs_retained': result['partitionRunCount'],
        'worker_partitions_skipped_by_gate': result['skippedWorkerPartitionCount'],
        'worker_runs_skipped_by_gate': result['skippedWorkerRunCount'],
    })
campaigns_df = pd.DataFrame(campaign_rows)


## 5. Optional: correlate failure patterns across the portfolio

This section is **not root-cause diagnosis**. It only asks whether the same stable error key appears in other campaigns or targets, which can help decide what to compare later.

Do not use a generic key such as `driver_exit` by itself to conclude that gh-aw or CAO is broken. First inspect the concrete Run metadata and linked evidence in the campaign drill-down below.


In [6]:
clusters = []
actions = []

if not errors_df.empty:
    for error_key, group in errors_df.groupby('error_key'):
        affected_campaigns = sorted(group.campaign.unique())
        affected_targets = sorted(group.target_repository.dropna().unique())
        latest = group.sort_values('latest_observed_at', ascending=False).iloc[0]

        if len(affected_campaigns) >= 2:
            scope = 'shared-platform'
            confidence = 'tentative'
            unresolved = 'The error identity may be generic; compare structured runtime and Audit dimensions.'
            action = 'Compare runtime/compiler versions and equivalent recent failures across campaigns.'
        else:
            campaign = affected_campaigns[0]
            campaign_group = group[group.campaign == campaign]
            if 'orchestrator' in set(campaign_group.role):
                scope = 'campaign-definition'
                confidence = 'tentative'
                unresolved = 'An orchestrator failure identifies an investigation boundary, not a root cause.'
                action = 'Inspect orchestrator setup, configuration, permissions, and its latest failed Run.'
            else:
                expected = expected_targets[campaign]
                evaluated = set(partitions_df[
                    (partitions_df.campaign == campaign)
                    & (partitions_df.role == 'worker')
                    & (partitions_df.target_scope_membership == 'expected')
                ].target_repository.dropna())
                affected = set(campaign_group.target_repository.dropna()) & expected
                complete = bool(expected) and expected <= evaluated
                if complete and len(expected) >= 2 and affected == expected:
                    scope = 'campaign-definition'
                    confidence = 'supported'
                    unresolved = None
                    action = 'Inspect common worker definition, campaign inputs, capabilities, and permissions.'
                elif complete and len(expected) >= 2 and affected and affected < expected:
                    scope = 'target-specific'
                    confidence = 'supported'
                    unresolved = None
                    action = 'Compare affected and unaffected repository permissions, content, and settings.'
                else:
                    scope = 'undetermined'
                    confidence = 'unknown'
                    unresolved = 'Expected-target coverage is unavailable or has fewer than two evaluated targets.'
                    action = 'Establish the expected-target denominator and reproduce on unevaluated targets.'

        clusters.append({
            'error_key': error_key,
            'diagnostic_scope': scope,
            'confidence': confidence,
            'affected_campaigns': len(affected_campaigns),
            'campaigns': ', '.join(affected_campaigns),
            'affected_partitions': len(group),
            'affected_targets': len(affected_targets),
            'latest_observed_at': latest.latest_observed_at,
            'latest_run_url': latest.latest_run_url,
            'likely_cause': 'unknown',
            'unresolved_question': unresolved,
        })
        actions.append({
            'error_key': error_key,
            'scope': scope,
            'state': 'needs-attention',
            'why': f'{len(group)} current error group(s) across {len(affected_campaigns)} campaign(s)',
            'next_action': action,
            'confidence': confidence,
            'latest_run_url': latest.latest_run_url,
        })

clusters_df = pd.DataFrame(clusters)
actions_df = pd.DataFrame(actions)


## 6. Inspect computation efficiency

The bridge benchmarks the shared JavaScript runtime in one Node process, avoiding subprocess startup noise between iterations.

The most useful architecture signals are empirical:

- **records visited**: Runs examined before each first-success boundary;
- **rows avoided**: retained history after those boundaries that did not need evaluation; and
- **worker Runs skipped by gate**: work avoided because orchestrator health was not trustworthy.

These measurements validate the computation shape. They are not a substitute for the dashboard’s separate `<200 ms` IndexedDB performance contract.


In [7]:
efficiency_rows = []
for row in campaigns_df.to_dict('records'):
    retained = int(row['partition_runs_retained'])
    visited = int(row['records_visited'])
    efficiency_rows.append({
        'campaign': row['campaign'],
        'orchestrator gate': row['worker_evaluation_state'],
        'evaluated partition Runs': retained,
        'records visited': visited,
        'rows avoided after success': max(retained - visited, 0),
        'worker Runs skipped by gate': int(row['worker_runs_skipped_by_gate']),
        'visit ratio %': round(100 * visited / retained, 1) if retained else 0,
    })

efficiency_df = pd.DataFrame(efficiency_rows)
runtime_summary = pd.DataFrame([{
    'measure': runtime_result['measureId'],
    'measure version': runtime_result['measureVersion'],
    'benchmark iterations': runtime_benchmark['iterations'],
    'median runtime ms': round(runtime_benchmark['medianMilliseconds'], 3),
    'minimum runtime ms': round(runtime_benchmark['minimumMilliseconds'], 3),
    'maximum runtime ms': round(runtime_benchmark['maximumMilliseconds'], 3),
    'evaluated partition Runs': int(campaigns_df.partition_runs_retained.sum()),
    'records visited': int(campaigns_df.records_visited.sum()),
    'rows avoided after success': int(
        campaigns_df.partition_runs_retained.sum() - campaigns_df.records_visited.sum()
    ),
    'worker Runs skipped by gate': int(campaigns_df.worker_runs_skipped_by_gate.sum()),
}])

display(Markdown('### Shared-runtime benchmark'))
display(runtime_summary)
display(Markdown('### Work by campaign'))
display(efficiency_df)
print('Interpret latency only as this in-memory runtime cost; validate production query and publication latency in IndexedDB performance tests.')


### Shared-runtime benchmark

,measure,measure version,benchmark iterations,median runtime ms,minimum runtime ms,maximum runtime ms,evaluated partition Runs,records visited,rows avoided after success,worker Runs skipped by gate
0,does-it-run,2.0.0,30,1.362,1.217,8.049,1843,465,1378,370


### Work by campaign

,campaign,orchestrator gate,evaluated partition Runs,records visited,rows avoided after success,worker Runs skipped by gate,visit ratio %
0,cao-evolution,eligible,87,7,80,0,8.0
1,dependabot,eligible,148,8,140,0,5.4
2,eslint-rules,eligible,14,2,12,0,14.3
3,eu-cra-compliance,blocked-by-orchestrator,17,2,15,369,11.8
4,optimization,eligible,1077,328,749,0,30.5
5,repo-assist,blocked-by-orchestrator,95,2,93,1,2.1
6,self-care,eligible,94,22,72,0,23.4
7,software-development-practices,eligible,210,86,124,0,41.0
8,uk-ai-advisory,eligible,101,8,93,0,7.9


Interpret latency only as this in-memory runtime cost; validate production query and publication latency in IndexedDB performance tests.


## 7. Read the portfolio overview

Start with campaign health and current attention partitions. Cross-campaign clusters and generic actions are optional triage aids; they do not explain an individual failure. Use the campaign drill-down in the next section for concrete Run metadata and evidence.


In [8]:
def display_answer_table(frame):
    html = frame.to_html(escape=True)
    html = html.replace(
        '<td>yes</td>',
        '<td style="background-color:#d1fae5;color:#065f46;font-weight:700">yes</td>',
    )
    html = html.replace(
        '<td>no</td>',
        '<td style="background-color:#fee2e2;color:#991b1b;font-weight:700">no</td>',
    )
    display(HTML(html))


answer_order = pd.CategoricalDtype(
    ['no', 'unknown', 'not-observed', 'running', 'yes'],
    ordered=True,
)
campaigns_view = campaigns_df.copy()
campaigns_view['answer'] = campaigns_view.answer.astype(answer_order)
campaigns_view = campaigns_view.sort_values(['answer', 'campaign'])

display(Markdown('### Campaign health'))
display_answer_table(campaigns_view)

columns = [
    'campaign',
    'workflow',
    'role',
    'target_repository',
    'target_scope_membership',
    'answer',
    'runs_since_success',
    'terminal_non_successes',
    'latest_run_at',
    'latest_run_url',
    'records_visited',
    'partition_runs_retained',
]
attention_df = partitions_df[partitions_df.needs_attention].sort_values(
    ['campaign', 'role', 'workflow', 'target_repository'],
    na_position='first',
)
display(Markdown('### Current attention partitions'))
display_answer_table(attention_df[columns].head(MAX_DISPLAY_ROWS))
if len(attention_df) > MAX_DISPLAY_ROWS:
    print(len(attention_df) - MAX_DISPLAY_ROWS, 'additional rows omitted from display only')

display(Markdown('### Cross-campaign failure clusters'))
display(clusters_df if not clusters_df.empty else Markdown('No current terminal failures.'))
display(Markdown('### Recommended next actions'))
display(actions_df if not actions_df.empty else Markdown('No actions currently justified.'))


### Campaign health

,campaign,answer,worker_evaluation_state,orchestrator_answer,worker_partitions_evaluated,attention_partitions,explicit_target_count,target_coverage,records_visited,partition_runs_retained,worker_partitions_skipped_by_gate,worker_runs_skipped_by_gate
1,dependabot,no,eligible,yes,2,1,1,not-assessed-without-dispatch-intent,8,148,0,0
3,eu-cra-compliance,no,blocked-by-orchestrator,no,0,1,0,not-assessed-without-dispatch-intent,2,17,43,369
4,optimization,no,eligible,yes,29,8,0,not-assessed-without-dispatch-intent,328,1077,0,0
5,repo-assist,no,blocked-by-orchestrator,no,0,1,0,not-assessed-without-dispatch-intent,2,95,1,1
6,self-care,no,eligible,yes,13,6,1,not-assessed-without-dispatch-intent,22,94,0,0
7,software-development-practices,no,eligible,yes,14,7,0,not-assessed-without-dispatch-intent,86,210,0,0
0,cao-evolution,yes,eligible,yes,6,0,0,not-assessed-without-dispatch-intent,7,87,0,0
2,eslint-rules,yes,eligible,yes,1,0,0,not-assessed-without-dispatch-intent,2,14,0,0
8,uk-ai-advisory,yes,eligible,yes,7,0,0,not-assessed-without-dispatch-intent,8,101,0,0


### Current attention partitions

,campaign,workflow,role,target_repository,target_scope_membership,answer,runs_since_success,terminal_non_successes,latest_run_at,latest_run_url,records_visited,partition_runs_retained
8,dependabot,dependabot-update-planner,worker,github/gh-aw,expected,no,5,5,2026-09-22T02:35:25.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35680021804,6,61
12,eu-cra-compliance,eu-cra-compliance,orchestrator,NaN,NaN,no,1,1,2026-09-16T19:51:41.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35143089588,2,17
29,optimization,optimization-agents-md-curator,worker,github/gh-aw-actions,unknown,no,3,3,2026-09-21T20:31:31.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35651605297,4,8
30,optimization,optimization-agents-md-curator,worker,github/gh-aw-firewall,unknown,no,1,1,2026-09-22T02:39:01.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35680234915,2,46
21,optimization,optimization-ai-credit-optimizer,worker,github/gh-aw,unknown,no,90,90,2026-09-22T02:38:23.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35680198271,90,90
22,optimization,optimization-ai-credit-optimizer,worker,github/gh-aw-actions,unknown,no,6,6,2026-09-21T20:31:38.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35651616055,6,6
23,optimization,optimization-ai-credit-optimizer,worker,github/gh-aw-firewall,unknown,no,50,50,2026-09-22T02:38:55.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35680228436,50,50
24,optimization,optimization-ai-credit-optimizer,worker,github/gh-aw-mcpg,unknown,no,30,30,2026-09-21T22:28:46.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35662874344,30,30
25,optimization,optimization-ai-credit-optimizer,worker,github/gh-aw-threat-detection,unknown,no,1,1,2026-09-21T03:37:23.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35558145018,1,1
42,optimization,optimization-token-optimizer,worker,NaN,unknown,no,123,123,2026-09-18T07:00:39.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35317413177,123,123


### Cross-campaign failure clusters

,error_key,diagnostic_scope,confidence,affected_campaigns,campaigns,affected_partitions,affected_targets,latest_observed_at,latest_run_url,likely_cause,unresolved_question
0,agent_logic,shared-platform,tentative,3,"optimization, self-care, software-development-practices",9,7,2026-09-21T16:34:55.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35626512832,unknown,The error identity may be generic; compare structured runtime and Audit dimensions.
1,baseline,shared-platform,tentative,2,"optimization, repo-assist",3,1,2026-09-22T02:38:23.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35680198271,unknown,The error identity may be generic; compare structured runtime and Audit dimensions.
2,driver_exit,shared-platform,tentative,4,"dependabot, eu-cra-compliance, optimization, self-care",14,6,2026-09-22T02:39:01.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35680234915,unknown,The error identity may be generic; compare structured runtime and Audit dimensions.


### Recommended next actions

,error_key,scope,state,why,next_action,confidence,latest_run_url
0,agent_logic,shared-platform,needs-attention,9 current error group(s) across 3 campaign(s),Compare runtime/compiler versions and equivalent recent failures across campaigns.,tentative,https://github.com/githubnext/gh-aw-cao/actions/runs/35626512832
1,baseline,shared-platform,needs-attention,3 current error group(s) across 2 campaign(s),Compare runtime/compiler versions and equivalent recent failures across campaigns.,tentative,https://github.com/githubnext/gh-aw-cao/actions/runs/35680198271
2,driver_exit,shared-platform,needs-attention,14 current error group(s) across 4 campaign(s),Compare runtime/compiler versions and equivalent recent failures across campaigns.,tentative,https://github.com/githubnext/gh-aw-cao/actions/runs/35680234915


## 8. Diagnose one campaign from the failing Run inward

A failed partition is an **investigation boundary**, not a root cause. The next step is to inspect the newest current failure and compare it with the partition’s latest successful Run.

`drilldown("campaign-slug")` follows this order:

1. confirm the campaign answer and orchestrator gate;
2. identify the failing workflow-target partition;
3. list current failure groups and all bounded Run references in the group;
4. load canonical metadata for those Runs from the SQLite projection;
5. inspect bounded Audit, tool-error, firewall, and safe-output records linked by canonical `runId`;
6. compare runtime versions with the latest successful Run; and
7. open the GitHub Actions Run when raw logs or stderr are required.

The local snapshot intentionally does **not** retain prompts, tool arguments, tool response bodies, transcripts, or complete Actions logs. Those can contain secrets and are not safe grouping dimensions. Consequently, metadata can narrow the diagnosis, but a `driver_exit` generally requires the linked Actions log to establish the exact process error.


In [9]:
_run_evidence_cache = {}
MAX_DIAGNOSTIC_RUNS = 5


def records_for_run(collection, canonical_run_id, limit=1000):
    return cao(
        'query',
        '--collection',
        collection,
        '--where',
        f'runId={canonical_run_id}',
        '--limit',
        str(limit),
    )


def run_evidence(github_run_id):
    key = str(int(github_run_id)) if isinstance(github_run_id, float) and github_run_id.is_integer() else str(github_run_id)
    if key in _run_evidence_cache:
        return _run_evidence_cache[key]

    matches = cao(
        'query',
        '--collection',
        'runs',
        '--where',
        f'githubRunId={key}',
        '--limit',
        '10',
    )
    if not matches:
        raise KeyError(f'Canonical Run {key} is not available in the SQLite projection')

    run = matches[0]
    canonical_run_id = run['id']
    evidence = {
        'run': run,
        'audits': records_for_run('audits', canonical_run_id),
        'tools': records_for_run('tools', canonical_run_id),
        'domains': records_for_run('domains', canonical_run_id),
        'issues': records_for_run('issues', canonical_run_id),
    }
    _run_evidence_cache[key] = evidence
    return evidence


def selected_run_metadata(run):
    fields = [
        'githubRunId',
        'startedAt',
        'completedAt',
        'duration',
        'status',
        'conclusion',
        'failureKind',
        'classification',
        'targetRepository',
        'rolloutMode',
        'ghAwVersion',
        'engine',
        'engineVersion',
        'agentVersion',
        'requestedModel',
        'resolvedModel',
        'firewallVersion',
        'firewallAllowedCalls',
        'firewallBlockedCalls',
        'mcpToolCalls',
        'highPriorityAuditItems',
        'mediumPriorityAuditItems',
        'errorCount',
        'runLink',
    ]
    return {field: run.get(field) for field in fields}


def failed_run_comparison(run_ids):
    rows = []
    for run_id in run_ids[:MAX_DIAGNOSTIC_RUNS]:
        rows.append(selected_run_metadata(run_evidence(run_id)['run']))
    return pd.DataFrame(rows)


def runtime_comparison(failed_run, successful_run):
    fields = [
        'ghAwVersion',
        'engine',
        'engineVersion',
        'agentVersion',
        'requestedModel',
        'resolvedModel',
        'firewallVersion',
    ]
    return pd.DataFrame([
        {
            'field': field,
            'failed Run': failed_run.get(field),
            'latest successful Run': successful_run.get(field),
            'same': failed_run.get(field) == successful_run.get(field),
        }
        for field in fields
    ])


def display_run_diagnosis(run_ids, latest_success_id=None):
    bounded_ids = [run_id for run_id in run_ids if run_id is not None][:MAX_DIAGNOSTIC_RUNS]
    if not bounded_ids:
        display(Markdown('No current failed Run reference is available.'))
        return

    display(Markdown('### Failure metadata — current failed Runs'))
    display(failed_run_comparison(bounded_ids))
    if len(run_ids) > MAX_DIAGNOSTIC_RUNS:
        print(len(run_ids) - MAX_DIAGNOSTIC_RUNS, 'additional current failed Runs omitted from diagnostic loading')

    evidence = run_evidence(bounded_ids[0])
    run = evidence['run']
    audits = evidence['audits']
    tools = evidence['tools']
    domains = evidence['domains']
    issues = evidence['issues']

    audit_rows = [{
        'timestamp': item.get('timestamp'),
        'source': item.get('source'),
        'type': item.get('type'),
        'status': item.get('status'),
        'summary': item.get('summary'),
        'error': item.get('error'),
    } for item in audits if (
        item.get('type') in {
            'audit.finding',
            'audit.recommendation',
            'workflow_run_assessment',
            'workflow_run_comparison',
            'workflow_run_failed',
            'workflow_run_grader',
        }
        or item.get('status') in {'critical', 'high', 'error', 'failure'}
    )]
    display(Markdown(f"### Audit and failure signals for latest Run `{bounded_ids[0]}`"))
    display(pd.DataFrame(audit_rows) if audit_rows else Markdown('No diagnostic Audit records are available.'))

    tool_errors = [item for item in tools if item.get('type') == 'tool.error']
    if tool_errors:
        tool_error_rows = pd.DataFrame([{
            'server': item.get('mcpServer'),
            'tool': item.get('mcpTool'),
            'status': item.get('status'),
            'first observed': item.get('timestamp'),
        } for item in tool_errors])
        tool_error_summary = (
            tool_error_rows
            .groupby(['server', 'tool', 'status'], dropna=False)
            .agg(observations=('tool', 'size'), first_observed=('first observed', 'min'))
            .reset_index()
            .sort_values(['observations', 'server', 'tool'], ascending=[False, True, True])
        )
    else:
        tool_error_summary = pd.DataFrame()
    display(Markdown('### Tool errors and incomplete-call observations'))
    display(tool_error_summary if not tool_error_summary.empty else Markdown('No tool-error observations are available.'))

    firewall_rows = pd.DataFrame([{
        'domain': item.get('domain'),
        'decision': item.get('decision'),
        'request_count': item.get('requestCount'),
    } for item in domains])
    display(Markdown('### Firewall observations'))
    display(firewall_rows if not firewall_rows.empty else Markdown('No firewall observations are available.'))

    display(Markdown('### Safe-output records'))
    display(pd.DataFrame(issues) if issues else Markdown('No safe-output issue records are available.'))

    grader_errors = [
        item for item in audits
        if item.get('type') == 'workflow_run_grader' and item.get('status') == 'error'
    ]
    blocked_requests = sum(
        int(item.get('requestCount') or 0)
        for item in domains
        if item.get('decision') == 'blocked'
    )
    diagnosis = pd.DataFrame([
        {
            'diagnostic question': 'What directly classified the Run as failed?',
            'observed evidence': f"{run.get('failureKind') or 'unknown'} / {run.get('classification') or 'unknown'}",
            'interpretation': 'This identifies the failure boundary, not the underlying exception.',
        },
        {
            'diagnostic question': 'Did the firewall block requests?',
            'observed evidence': f'{blocked_requests} blocked request(s)',
            'interpretation': 'No blocks weakens a firewall-block hypothesis; it does not prove all permissions were correct.',
        },
        {
            'diagnostic question': 'Did tools report incomplete/error observations?',
            'observed evidence': ', '.join(sorted({item.get('summary') or 'unknown' for item in tool_errors})) or 'none',
            'interpretation': 'Correlated tool evidence is a candidate clue; inspect timestamps and the Actions log before assigning cause.',
        },
        {
            'diagnostic question': 'Did a post-run grader also fail?',
            'observed evidence': f'{len(grader_errors)} grader error(s)',
            'interpretation': 'A grader error is a separate diagnostic-quality problem and may be secondary to the workflow failure.',
        },
        {
            'diagnostic question': 'Is raw stderr or complete output retained here?',
            'observed evidence': 'no',
            'interpretation': 'Open the Actions Run to determine the exact driver-exit cause.',
        },
    ])
    display(Markdown('### What this metadata suggests — not yet root cause'))
    display(diagnosis)

    if latest_success_id is not None:
        successful = run_evidence(latest_success_id)['run']
        display(Markdown(f"### Runtime comparison with latest success `{successful['githubRunId']}`"))
        display_answer_table(runtime_comparison(run, successful))

    run_link = run.get('runLink')
    if run_link:
        display(Markdown(f'**Next evidence source:** [Open GitHub Actions Run {bounded_ids[0]}]({run_link}) for job logs and stderr.'))


def drilldown(campaign):
    known_campaigns = set(campaigns_df.campaign)
    if campaign not in known_campaigns:
        raise KeyError(f'Choose from {sorted(known_campaigns)}')

    display(Markdown(f'## `{campaign}`'))
    display_answer_table(campaigns_df[campaigns_df.campaign == campaign])

    campaign_partitions = partitions_df[partitions_df.campaign == campaign].sort_values(
        ['role', 'workflow', 'target_repository'],
        na_position='first',
    )
    display(Markdown('### Evaluated partitions'))
    display_answer_table(campaign_partitions[columns])

    campaign_errors = (
        errors_df[errors_df.campaign == campaign]
        if not errors_df.empty
        else pd.DataFrame()
    )
    display(Markdown('### Current error groups'))
    if campaign_errors.empty:
        display(Markdown('No current error groups.'))
    else:
        visible_error_columns = [
            'campaign',
            'workflow',
            'role',
            'target_repository',
            'target_scope_membership',
            'error_key',
            'count',
            'latest_observed_at',
            'latest_run_id',
            'latest_run_url',
        ]
        display(campaign_errors[visible_error_columns])

    if not campaign_errors.empty:
        display(Markdown('**Next: inspect the concrete failure metadata below.** The error group selects related Runs; it does not explain their cause.'))

    for error in campaign_errors.head(MAX_DIAGNOSTIC_RUNS).to_dict('records'):
        matching_partition = campaign_partitions[
            (campaign_partitions.workflow == error['workflow'])
            & (campaign_partitions.target_repository.fillna('') == (error['target_repository'] or ''))
        ]
        latest_success_id = (
            matching_partition.latest_success_id.iloc[0]
            if len(matching_partition) and pd.notna(matching_partition.latest_success_id.iloc[0])
            else None
        )
        display_run_diagnosis(error['current_run_ids'], latest_success_id)

    if not campaign_errors.empty and not clusters_df.empty:
        error_keys = set(campaign_errors.error_key)
        display(Markdown('### Optional: broader portfolio correlation'))
        display(Markdown('This table only identifies other partitions worth comparing. It is not a root-cause conclusion.'))
        display(clusters_df[clusters_df.error_key.isin(error_keys)])


attention_campaigns = campaigns_view[
    campaigns_view.answer.isin(['no', 'unknown', 'not-observed'])
].campaign
example_campaign = (
    'dependabot'
    if 'dependabot' in set(attention_campaigns)
    else attention_campaigns.iloc[0]
    if len(attention_campaigns)
    else campaigns_view.campaign.iloc[0]
)
drilldown(example_campaign)


## `dependabot`

,campaign,answer,worker_evaluation_state,orchestrator_answer,worker_partitions_evaluated,attention_partitions,explicit_target_count,target_coverage,records_visited,partition_runs_retained,worker_partitions_skipped_by_gate,worker_runs_skipped_by_gate
1,dependabot,no,eligible,yes,2,1,1,not-assessed-without-dispatch-intent,8,148,0,0


### Evaluated partitions

,campaign,workflow,role,target_repository,target_scope_membership,answer,runs_since_success,terminal_non_successes,latest_run_at,latest_run_url,records_visited,partition_runs_retained
7,dependabot,dependabot,orchestrator,NaN,NaN,yes,0,0,2026-09-22T02:31:16.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35679760927,1,84
8,dependabot,dependabot-update-planner,worker,github/gh-aw,expected,no,5,5,2026-09-22T02:35:25.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35680021804,6,61
9,dependabot,dependabot-update-planner,worker,github/gh-aw-firewall,observed-extra,yes,0,0,2026-09-22T01:36:04.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35676360221,1,3


### Current error groups

...

,field,failed Run,latest successful Run,same
0,ghAwVersion,v0.89.17,v0.89.17,True
1,engine,GitHub Copilot CLI,GitHub Copilot CLI,True
2,engineVersion,1.0.85,1.0.85,True
3,agentVersion,1.0.85,1.0.85,True
4,requestedModel,auto,auto,True
5,resolvedModel,auto,auto,True
6,firewallVersion,v0.28.20,v0.28.20,True


**Next evidence source:** [Open GitHub Actions Run 35680021804](https://github.com/githubnext/gh-aw-cao/actions/runs/35680021804) for job logs and stderr.

### Optional: broader portfolio correlation

This table only identifies other partitions worth comparing. It is not a root-cause conclusion.

,error_key,diagnostic_scope,confidence,affected_campaigns,campaigns,affected_partitions,affected_targets,latest_observed_at,latest_run_url,likely_cause,unresolved_question
2,driver_exit,shared-platform,tentative,4,"dependabot, eu-cra-compliance, optimization, self-care",14,6,2026-09-22T02:39:01.000Z,https://github.com/githubnext/gh-aw-cao/actions/runs/35680234915,unknown,The error identity may be generic; compare structured runtime and Audit dimensions.


## Interpretation and production boundary

- Campaign answers are constrained by the JSONL enrichment coverage shown at the top.
- The orchestrator gate prevents worker evidence from being trusted when dispatch itself currently fails or is unknown.
- Worker success is isolated by target; one repository cannot reset another repository’s failure streak.
- Review-mode campaigns without explicit targets expect the CAO control repository. Other observed repositories remain visible but do not decide configured campaign health.
- A failure group answers **which Runs should be investigated together**; it does not prove why they failed.
- Canonical Run metadata can expose failure classifications, versions, models, timing, resource use, Audit findings, grader failures, tool-error observations, firewall decisions, and safe-output records.
- Raw prompts, tool arguments, tool response bodies, transcripts, full logs, and stderr are intentionally absent from the derived snapshot. The GitHub Actions Run remains the evidence source for an exact process-level failure.
- The notebook adapter reads local JSONL and queries local SQLite for exploration. Production selection, filtering, ordering, joins, and partition construction remain Dashboard Language/data-worker responsibilities.
- The shared runtime consumes preselected, newest-first partitions and owns only versioned `does-it-run@2.0.0` semantics.
